# Lab 4: Logical Planning — Using an LLM to Construct and Test a Simple Planning Agent

## Objective

This laboratory focuses on logical planning for a warehouse robot. The planning problem is represented using states, actions, preconditions, and effects.

The laboratory covers:

- Formulating a planning problem using logical propositions
- Determining whether actions are applicable in a given state
- Constructing a valid plan manually
- Implementing a planning agent using an LLM
- Using Breadth-First Search to explore possible plans
- Testing and validating the generated planner
- Understanding how logical reasoning and search work together
- Evaluating the use of an LLM as an engineering assistant
- Understanding independent verification of AI-generated plans

## Task 0: Specify the Planning Problem

Before implementing the planner, we first represent the warehouse problem using states, actions, preconditions, and effects.

### Initial State

The robot starts at location A, and the package is also at location A.

\[
I = \{At(Robot,A),\ At(Package,A)\}
\]

### Goal State

The objective is to move the package to location C.

\[
G = \{At(Package,C)\}
\]

### Available Actions

#### 1. Move

The robot can move between connected locations.

Example:

**Move(A,B)**

**Precondition:**
- `At(Robot,A)`

**Effects:**
- Remove `At(Robot,A)`
- Add `At(Robot,B)`

#### 2. PickUp

The robot can pick up the package when both are at the same location.

Example:

**PickUp(Package,A)**

**Preconditions:**
- `At(Robot,A)`
- `At(Package,A)`

**Effects:**
- Remove `At(Package,A)`
- Add `Holding(Package)`

#### 3. Drop

The robot can drop the package at its current location when it is holding it.

Example:

**Drop(Package,C)**

**Preconditions:**
- `At(Robot,C)`
- `Holding(Package)`

**Effects:**
- Remove `Holding(Package)`
- Add `At(Package,C)`

### Initially Applicable Actions

The initial state is:

\[
S_0 = \{At(Robot,A),\ At(Package,A)\}
\]

An action is applicable only when **all of its preconditions are satisfied** in the current state.

- **PickUp(Package,A): Applicable**
  - `At(Robot,A)` is true.
  - `At(Package,A)` is true.
  - Therefore, all preconditions are satisfied.

- **Drop(Package,C): Not applicable**
  - It requires `At(Robot,C)`, but the robot is initially at A.
  - It also requires `Holding(Package)`, which is false initially.

Therefore, **PickUp(Package,A) is initially applicable, while Drop(Package,C) is not.**

### Think About It

An action should not be considered applicable just because it is available in the action list. Its preconditions must be satisfied in the current state.

For example, in the initial state:

`{At(Robot,A), At(Package,A)}`

`PickUp(Package,A)` is applicable because both the robot and package are at A.

However, `Drop(Package,C)` is not applicable because the robot is not at C and is not holding the package.

Therefore, logical reasoning is used to check whether all the required preconditions of an action are satisfied before applying it.

## Task 1: Manually Construct the Plan

We need to transport the package from location A to location C.

The warehouse locations are connected as:

\[
A \leftrightarrow B \leftrightarrow C
\]

The robot must first pick up the package, then move to C while holding it, and finally drop it.

### Initial State

\[
S_0 = \{At(Robot,A),\ At(Package,A)\}
\]

### Action 1: PickUp(Package,A)

Preconditions:
- `At(Robot,A)` ✓
- `At(Package,A)` ✓

Both preconditions are satisfied, so the action is applicable.

Effects:
- Remove `At(Package,A)`
- Add `Holding(Package)`

State after Action 1:

\[
S_1 = \{At(Robot,A),\ Holding(Package)\}
\]

---

### Action 2: Move(A,B)

Precondition:
- `At(Robot,A)` ✓

The action is applicable.

Effects:
- Remove `At(Robot,A)`
- Add `At(Robot,B)`

State after Action 2:

\[
S_2 = \{At(Robot,B),\ Holding(Package)\}
\]

---

### Action 3: Move(B,C)

Precondition:
- `At(Robot,B)` ✓

The action is applicable.

Effects:
- Remove `At(Robot,B)`
- Add `At(Robot,C)`

State after Action 3:

\[
S_3 = \{At(Robot,C),\ Holding(Package)\}
\]

---

### Action 4: Drop(Package,C)

Preconditions:
- `At(Robot,C)` ✓
- `Holding(Package)` ✓

Both preconditions are satisfied, so the action is applicable.

Effects:
- Remove `Holding(Package)`
- Add `At(Package,C)`

Final state:

\[
S_4 = \{At(Robot,C),\ At(Package,C)\}
\]

The goal

\[
G = \{At(Package,C)\}
\]

is satisfied.

### Final Plan

\[
\boxed{
PickUp(Package,A)
\rightarrow
Move(A,B)
\rightarrow
Move(B,C)
\rightarrow
Drop(Package,C)
}
\]

Therefore, the manually constructed plan successfully transports the package from A to C.

## Task 2: Construct a Planning Agent Using an LLM

We now use an LLM to generate a Python implementation of the planning agent.

Instead of asking the LLM to simply solve the warehouse problem, we provide a precise specification of how states, actions, preconditions, effects, and search should be represented.

### LLM Prompt

The prompt used to generate the planner was:

> Write a Python planning agent for a simple warehouse robot problem.
>
> Represent a state as a set of logical propositions.
>
> Represent each action using:
> - an action name,
> - positive preconditions,
> - negative preconditions,
> - positive effects,
> - negative effects.
>
> An action is applicable if all its preconditions are satisfied in the current state.
>
> To apply an action:
> - remove its negative effects from the current state,
> - add its positive effects to the current state.
>
> Use Breadth-First Search (BFS) to find a sequence of actions that transforms the initial state into a state satisfying the goal.
>
> The planner should:
> - return a valid sequence of actions if a plan exists,
> - report that no plan was found if the goal is unreachable,
> - print the actions in the resulting plan,
> - print the state after each action.
>
> Use the warehouse example with locations A, B, and C:
> - The robot starts at A.
> - The package starts at A.
> - The goal is for the package to be at C.
> - The robot can move between A and B, and between B and C.
> - The robot can pick up the package when both are at the same location.
> - The robot can drop the package at its current location when holding it.
>
> Also explain the implementation and state any assumptions made.

### Planner Logic

The planning process can be summarized as:

1. Start from the initial state.
2. Check which actions have satisfied preconditions.
3. Apply an applicable action to generate a successor state.
4. Use BFS to explore different possible action sequences.
5. Continue until a state satisfying the goal is reached.
6. If no reachable state satisfies the goal, report that no plan was found.

The key idea is:

**Logic determines what is possible; search determines what to try.**

In [29]:
from collections import deque

class Action:
    def __init__(self, name, positive_preconditions,
                 negative_preconditions, positive_effects,
                 negative_effects):
        self.name = name
        self.positive_preconditions = set(positive_preconditions)
        self.negative_preconditions = set(negative_preconditions)
        self.positive_effects = set(positive_effects)
        self.negative_effects = set(negative_effects)

    def is_applicable(self, state):
        return (
            self.positive_preconditions.issubset(state)
            and self.negative_preconditions.isdisjoint(state)
        )

    def apply(self, state):
        new_state = set(state)

        new_state -= self.negative_effects

        new_state |= self.positive_effects

        return new_state

def bfs_planner(initial_state, goal, actions):

    initial_state = frozenset(initial_state)

    queue = deque()
    queue.append((initial_state, []))

    visited = {initial_state}

    while queue:

        current_state, plan = queue.popleft()

        if goal.issubset(current_state):
            return plan

        for action in actions:

            if action.is_applicable(current_state):

                next_state = frozenset(action.apply(current_state))

                if next_state not in visited:
                    visited.add(next_state)

                    new_plan = plan + [action]
                    queue.append((next_state, new_plan))

    return None

initial_state = {
    "robot_at_A",
    "package_at_A",
    "not_holding"
}


goal = {
    "package_at_C"
}

actions = [

    Action(
        name="Move A -> B",
        positive_preconditions={"robot_at_A"},
        negative_preconditions=set(),
        positive_effects={"robot_at_B"},
        negative_effects={"robot_at_A"}
    ),

    Action(
        name="Move B -> A",
        positive_preconditions={"robot_at_B"},
        negative_preconditions=set(),
        positive_effects={"robot_at_A"},
        negative_effects={"robot_at_B"}
    ),

    Action(
        name="Move B -> C",
        positive_preconditions={"robot_at_B"},
        negative_preconditions=set(),
        positive_effects={"robot_at_C"},
        negative_effects={"robot_at_B"}
    ),

    Action(
        name="Move C -> B",
        positive_preconditions={"robot_at_C"},
        negative_preconditions=set(),
        positive_effects={"robot_at_B"},
        negative_effects={"robot_at_C"}
    ),

    Action(
        name="Pick up package at A",
        positive_preconditions={
            "robot_at_A",
            "package_at_A",
            "not_holding"
        },
        negative_preconditions=set(),
        positive_effects={"holding"},
        negative_effects={
            "package_at_A",
            "not_holding"
        }
    ),

    Action(
        name="Pick up package at B",
        positive_preconditions={
            "robot_at_B",
            "package_at_B",
            "not_holding"
        },
        negative_preconditions=set(),
        positive_effects={"holding"},
        negative_effects={
            "package_at_B",
            "not_holding"
        }
    ),

    Action(
        name="Pick up package at C",
        positive_preconditions={
            "robot_at_C",
            "package_at_C",
            "not_holding"
        },
        negative_preconditions=set(),
        positive_effects={"holding"},
        negative_effects={
            "package_at_C",
            "not_holding"
        }
    ),

    Action(
        name="Drop package at A",
        positive_preconditions={
            "robot_at_A",
            "holding"
        },
        negative_preconditions=set(),
        positive_effects={
            "package_at_A",
            "not_holding"
        },
        negative_effects={"holding"}
    ),

    Action(
        name="Drop package at B",
        positive_preconditions={
            "robot_at_B",
            "holding"
        },
        negative_preconditions=set(),
        positive_effects={
            "package_at_B",
            "not_holding"
        },
        negative_effects={"holding"}
    ),

    Action(
        name="Drop package at C",
        positive_preconditions={
            "robot_at_C",
            "holding"
        },
        negative_preconditions=set(),
        positive_effects={
            "package_at_C",
            "not_holding"
        },
        negative_effects={"holding"}
    )
]

plan = bfs_planner(initial_state, goal, actions)

if plan is None:

    print("No plan was found.")

else:

    print("Plan found!")
    print("Number of actions:", len(plan))
    print("\nActions and resulting states:")

    current_state = set(initial_state)

    print("\nInitial state:")
    print(sorted(current_state))

    for i, action in enumerate(plan, start=1):

        current_state = action.apply(current_state)

        print(f"\nStep {i}: {action.name}")
        print("State:")
        print(sorted(current_state))

Plan found!
Number of actions: 4

Actions and resulting states:

Initial state:
['not_holding', 'package_at_A', 'robot_at_A']

Step 1: Pick up package at A
State:
['holding', 'robot_at_A']

Step 2: Move A -> B
State:
['holding', 'robot_at_B']

Step 3: Move B -> C
State:
['holding', 'robot_at_C']

Step 4: Drop package at C
State:
['not_holding', 'package_at_C', 'robot_at_C']


### Think About It

The main ideas from the specification appear in the generated Python program as follows:

- **Preconditions → When is an action applicable?**

  Preconditions are checked by the `is_applicable()` function. It verifies that all positive preconditions are present in the current state and that all negative preconditions are absent.

- **Effects → How does the state change?**

  The `apply()` function removes the negative effects from the current state and adds the positive effects to produce the new state.

- **Goal → When does planning terminate?**

  The BFS planner checks whether the current state satisfies the goal. If the goal is satisfied, the planner returns the current sequence of actions.

- **BFS → How are alternative plans explored?**

  The `bfs_planner()` function uses a queue to explore states level by level. Applicable actions generate new states, which are added to the queue and explored until a goal state is found.

Thus, the Python implementation follows the main requirements given in the LLM prompt.

## Task 3: Testing and Validation

The generated planner should not be assumed to be correct simply because it produces a reasonable-looking plan.

We will test it using three cases:

1. **Test A — Solvable problem:** Verify that the planner reaches the goal.
2. **Test B — Impossible problem:** Remove the PickUp action and verify that no plan is found.
3. **Test C — Irrelevant actions:** Add an action that moves the robot but does not move the package, and verify that reaching location C with the robot is not incorrectly treated as achieving the package goal.

### Test A: Solvable Warehouse Problem

The original warehouse problem was tested with the complete set of actions.

The planner found a plan containing four actions:

1. Pick up package at A
2. Move A -> B
3. Move B -> C
4. Drop package at C

The final state was:

`{not_holding, package_at_C, robot_at_C}`

Since `package_at_C` is present in the final state, the goal was achieved.

The returned plan is valid because each action's preconditions are satisfied in the state before that action is executed.

### Test B: Impossible Problem

To create an impossible planning problem, we remove the `Pick up package at A` action.

Without the PickUp action, the robot can move between locations, but the package remains at A.

Therefore, the robot may reach C, but it can never obtain the package and move it to C.

The expected result is:

`No plan was found.`

In [30]:
actions_without_pickup = [
    action for action in actions
    if not action.name.startswith("Pick up")
]

plan_test_b = bfs_planner(
    initial_state,
    goal,
    actions_without_pickup
)

if plan_test_b is None:
    print("No plan was found.")
else:
    print("Plan found!")
    for action in plan_test_b:
        print(action.name)

No plan was found.


### Test C: Irrelevant Actions

We now add an irrelevant action that allows the robot to move directly from A to C.

This action changes only the robot's location. It does not move the package.

The purpose of this test is to verify that the planner checks the actual goal condition `package_at_C` rather than incorrectly treating `robot_at_C` as the goal.

The expected behavior is that the planner should still use the package manipulation actions and should not consider merely reaching C with the robot as achieving the goal.

In [31]:
irrelevant_action = Action(
    name="Move A -> C (irrelevant)",
    positive_preconditions={"robot_at_A"},
    negative_preconditions=set(),
    positive_effects={"robot_at_C"},
    negative_effects={"robot_at_A"}
)

actions_with_irrelevant = actions + [irrelevant_action]

plan_test_c = bfs_planner(
    initial_state,
    goal,
    actions_with_irrelevant
)

if plan_test_c is None:
    print("No plan was found.")
else:
    print("Plan found!")
    print("Number of actions:", len(plan_test_c))

    for i, action in enumerate(plan_test_c, start=1):
        print(f"Step {i}: {action.name}")

Plan found!
Number of actions: 3
Step 1: Pick up package at A
Step 2: Move A -> C (irrelevant)
Step 3: Drop package at C


### Test C: Results and Validation

The planner found the following plan:

1. Pick up package at A
2. Move A -> C (irrelevant)
3. Drop package at C

The additional `Move A -> C` action moves only the robot and does not move the package.

The planner did not treat `robot_at_C` as satisfying the goal because the actual goal is `package_at_C`.

After picking up the package, the robot can move directly from A to C using the added action. The package remains held by the robot, so dropping it at C produces `package_at_C`.

Therefore, the planner correctly distinguishes between the robot's location and the package's location.

## Task 4: Logic vs Search

Planning combines logical reasoning with search.

### Role of Logic

Logic determines whether an action is possible in the current state.

In our planner, this is handled by checking:

`action.is_applicable(current_state)`

The function checks whether:

- all positive preconditions are present in the current state, and
- all negative preconditions are absent from the current state.

If the preconditions are satisfied, the action is applicable.

Logic is also used when applying an action. The negative effects are removed from the current state and the positive effects are added to generate the successor state.

### Role of Search

Search determines which sequence of applicable actions should be explored.

Our planner uses **Breadth-First Search (BFS)**.

BFS:

1. Starts from the initial state.
2. Checks the available actions.
3. Finds the applicable actions.
4. Generates their resulting states.
5. Explores the resulting states using a queue.
6. Continues until a state satisfying the goal is found.

Therefore:

> **Logic determines what is possible; search determines what to try.**

### Overall Planning Process

The planning process can be represented as:

**Current State**  
↓  
**Check Action Preconditions**  
↓  
**Select an Applicable Action**  
↓  
**Generate Successor State**  
↓  
**Search Over Alternative Actions**  
↓  
**Check Goal**  
↓  
**Goal Reached → Return Plan**

### Think About It

The planner combines logical reasoning and search to produce a valid plan.

For a given state, logical reasoning first checks whether an action's preconditions are satisfied. If they are satisfied, the action is possible and can be applied to generate a successor state.

The search algorithm then explores different sequences of these applicable actions. In our implementation, BFS explores the states level by level using a queue.

Therefore:

**Logic determines what is possible; search determines what to try.**

The overall process is:

**Current State**
↓
**Check Action Preconditions**
↓
**Select an Applicable Action**
↓
**Generate Successor State**
↓
**Search Over Alternatives**
↓
**Check Goal**
↓
**Goal Reached → Return Plan**

Thus, planning combines the logical reasoning used to determine valid actions with the search process used to find a sequence of actions that reaches the goal.

## Task 5: LLM Explanation and Independent Verification

The planner generated the following plan for the original warehouse problem:

1. Pick up package at A
2. Move A -> B
3. Move B -> C
4. Drop package at C

We now verify why each action is valid by checking its preconditions against the state before the action.

### Action 1: Pick up package at A

State before action:

`{robot_at_A, package_at_A, not_holding}`

Required preconditions:

- `robot_at_A` ✓
- `package_at_A` ✓
- `not_holding` ✓

Therefore, the action is applicable.

Resulting state:

`{robot_at_A, holding}`

---

### Action 2: Move A -> B

State before action:

`{robot_at_A, holding}`

Required precondition:

- `robot_at_A` ✓

Therefore, the action is applicable.

Resulting state:

`{robot_at_B, holding}`

---

### Action 3: Move B -> C

State before action:

`{robot_at_B, holding}`

Required precondition:

- `robot_at_B` ✓

Therefore, the action is applicable.

Resulting state:

`{robot_at_C, holding}`

---

### Action 4: Drop package at C

State before action:

`{robot_at_C, holding}`

Required preconditions:

- `robot_at_C` ✓
- `holding` ✓

Therefore, the action is applicable.

Resulting state:

`{robot_at_C, package_at_C, not_holding}`

The goal `package_at_C` is now satisfied.

### Independent Verification

The LLM explanation describes why the plan should be valid, but the explanation itself is not sufficient evidence that the planner is correct.

The Python program independently executed each action and generated the resulting states. We verified that:

- every action's preconditions were satisfied before execution;
- each action produced the expected state changes; and
- the final state contained `package_at_C`.

Therefore, the validity of the plan was independently verified using the actual state transitions produced by the program.

This demonstrates an important engineering principle:

> **A generated explanation is not a substitute for independent verification.**

## Task 6: Prolog Representation of the Warehouse

For the optional Prolog extension, the warehouse connectivity is represented using Prolog facts and rules in the file `planner.pl`.

The warehouse connections are represented by the following facts:

- `connected(a,b)`
- `connected(b,a)`
- `connected(b,c)`
- `connected(c,b)`

The rule

`can_move(X,Y) :- connected(X,Y).`

means that the robot can move from location X to location Y if X and Y are directly connected.

The Prolog file was loaded and the following queries were tested.

For the query:

`can_move(a,b).`

the result is `true` because A and B are directly connected.

For the query:

`can_move(a,c).`

the result is `false` because A and C are not directly connected.

Although the robot can travel from A to C through B, there is no direct connection between A and C.

In [32]:
!apt-get update -qq
!apt-get install -y swi-prolog

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swi-prolog is already the newest version (9.0.4+dfsg-3.1ubuntu4).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.


In [33]:
!swipl --version

SWI-Prolog version 9.0.4 for x86_64-linux


In [34]:
!ls -l planner.pl

-rw-r--r-- 1 root root 210 Oct  1 16:06 planner.pl


In [35]:
print("Task 6: Prolog Warehouse Queries")

!swipl -q -s planner.pl -g "(can_move(a,b) -> writeln('can_move(a,b) = true'); writeln('can_move(a,b) = false')), halt."

!swipl -q -s planner.pl -g "(can_move(a,c) -> writeln('can_move(a,c) = true'); writeln('can_move(a,c) = false')), halt."

Task 6: Prolog Warehouse Queries
can_move(a,b) = true
can_move(a,c) = false


## Task 7: Verify a Proposed Plan

We use the `valid_move(X,Y)` rule defined in `planner.pl` to verify a proposed movement plan.

The proposed plan is:

`Move(a,b)` followed by `Move(b,c)`.

First, we check `valid_move(a,b)`. Since A and B are directly connected, the result should be `true`.

Next, we check `valid_move(b,c)`. Since B and C are directly connected, the result should also be `true`.

Finally, we test `valid_move(a,c)`. Since A and C are not directly connected, the result should be `false`.

Therefore, the proposed movement sequence `Move(a,b) → Move(b,c)` is valid, while `Move(a,c)` is not a valid direct movement.

In [36]:
print("Task 7: Proposed Plan Verification")

!swipl -q -s planner.pl -g "(valid_move(a,b) -> writeln('Move(a,b) = valid'); writeln('Move(a,b) = invalid')), halt."

!swipl -q -s planner.pl -g "(valid_move(b,c) -> writeln('Move(b,c) = valid'); writeln('Move(b,c) = invalid')), halt."

!swipl -q -s planner.pl -g "(valid_move(a,c) -> writeln('Move(a,c) = valid'); writeln('Move(a,c) = invalid')), halt."

Task 7: Proposed Plan Verification
Move(a,b) = valid
Move(b,c) = valid
Move(a,c) = invalid


## Task 8: Prolog Rule Chaining

Prolog can perform logical reasoning by chaining facts and rules.

The `planner.pl` file contains the fact:

`wet_road.`

This states that the road is wet.

It also contains the rule:

`slippery :- wet_road.`

This means that if the road is wet, then the road is slippery.

The next rule is:

`reduce_speed :- slippery.`

This means that if the road is slippery, then the vehicle should reduce its speed.

Therefore, Prolog can derive the following reasoning chain:

`wet_road → slippery → reduce_speed`

We can query each statement to verify the reasoning.

If `wet_road` is true, Prolog can derive `slippery`.

Since `slippery` is true, Prolog can then derive `reduce_speed`.

Thus, the final query `reduce_speed` should return `true`.

In [37]:
print("Task 8: Prolog Rule Chaining")

!swipl -q -s planner.pl -g "(wet_road -> writeln('wet_road = true'); writeln('wet_road = false')), halt."

!swipl -q -s planner.pl -g "(slippery -> writeln('slippery = true'); writeln('slippery = false')), halt."

!swipl -q -s planner.pl -g "(reduce_speed -> writeln('reduce_speed = true'); writeln('reduce_speed = false')), halt."

Task 8: Prolog Rule Chaining
wet_road = true
slippery = true
reduce_speed = true


# Reflection

## Main Laboratory Reflection

### 1. Why is it useful to specify action preconditions and effects before asking an LLM to write the planner?

Specifying preconditions and effects clearly tells the LLM exactly when an action can be used and how it changes the state. This reduces ambiguity and helps the generated planner follow the intended planning model.

### 2. Give an example of an error that could occur if the planner failed to check an action's preconditions.

The planner might try to drop the package at C even when the robot is not at C or is not holding the package. This would produce an invalid plan because the required conditions for the action are not satisfied.

### 3. Why is a plan that "looks reasonable" not necessarily a valid plan?

A plan may appear reasonable but contain an action whose preconditions are not satisfied. A valid plan must satisfy the preconditions of every action and produce the correct state transitions until the goal is reached.

### 4. What did the LLM contribute to the implementation?

The LLM generated the Python implementation of the planning agent, including the action representation, precondition checking, state transitions, and BFS-based search.

### 5. What did you have to verify independently?

I independently verified that the generated plan satisfied the action preconditions, that each action produced the expected state, that the final state satisfied the goal, and that the planner correctly handled the additional test cases.

### 6. In this laboratory, where is logical reasoning being used?

Logical reasoning is used when checking whether an action's preconditions are satisfied and when determining how an action's effects change the current state. It is also used in the Prolog extension through facts, rules, and queries.

### 7. How is planning related to the search algorithms studied in the previous module?

Planning uses search to explore different sequences of actions. Each state is a search state, and applicable actions generate successor states. In this laboratory, BFS was used to search for a sequence of actions that reaches the goal.

## Prolog Reflection

### 1. What is the difference between a Prolog fact and a Prolog rule?

A Prolog fact directly states something that is true, such as `connected(a,b).` A Prolog rule defines a condition under which a new statement can be concluded, such as `can_move(X,Y) :- connected(X,Y).`

### 2. How does a Prolog query correspond to asking whether something follows from a knowledge base?

A Prolog query asks whether a particular statement can be established using the facts and rules in the knowledge base. For example, `can_move(a,b)` is true because the knowledge base contains `connected(a,b)`.

### 3. Why might it be useful to use a Prolog program to verify a plan generated by a Python program?

Prolog provides an independent logical representation of the problem. It can check whether the movements in a proposed plan follow the known facts and rules, providing a separate way to verify the plan.

### 4. What advantage does an independent verifier provide when the original plan was generated with the help of an LLM?

An independent verifier provides an additional check that does not rely on the LLM's explanation being correct. The LLM can generate a candidate solution, while the separate logical system can check whether the solution follows the specified knowledge and rules.

### Key Idea

**An AI system can generate a candidate solution, while a separate logical system checks it.**

## Reflection on the Use of the LLM

The LLM contributed to the implementation by generating the Python planning agent from the specification provided in the prompt. It helped implement the `Action` class, precondition checking, state transitions, and the BFS-based planner.

However, the generated code could not be accepted without testing. I independently verified that the planner correctly checked action preconditions, applied the effects correctly, reached the required goal, and handled cases where no valid plan existed.

I also tested the planner with the original solvable problem, an impossible problem where the PickUp action was removed, and a modified problem containing an additional robot movement action.

The Prolog extension provided another independent way to verify logical reasoning. The Python planner generated and searched for a plan, while the Prolog program was used to verify movement rules and demonstrate logical rule chaining.

This laboratory showed that an LLM can assist with implementation, but the generated code and its output still need to be tested and independently verified.